In [12]:
import pandas as pd
import scrapy

In [2]:
!scrapy startproject news_scraper

New Scrapy project 'news_scraper', using template directory 'C:\Users\rebec\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\scrapy\templates\project', created in:
    C:\Users\rebec\OneDrive - Universidad de Oviedo\Escritorio\TFG\news_scraper

You can start your first spider with:
    cd news_scraper
    scrapy genspider example example.com


In [3]:
%cd news_scraper

c:\Users\rebec\OneDrive - Universidad de Oviedo\Escritorio\TFG\news_scraper


In [9]:
%%writefile "sitemap_spider.py"
import scrapy
class SitemapSpider(scrapy.Spider):
    name = 'sitemap_spider'
    start_urls = ['https://www.elcomercio.es/sitemap.xml',
                  'https://www.lne.es/sitemap_google_news_52e19.xml',
                  'https://www.larazon.es/sitemaps/news.xml',
                  'https://www.lavanguardia.com/sitemap-google-news.xml',
                  'https://www.rtpa.es/sitemap-noticias.xml',
                  'https://www.europapress.es/news_sitemap_1.xml',
                  'https://www.abc.es/sitemap.xml',
                  'https://www.20minutos.es/sitemap-google-news.xml',
                  'https://www.elperiodico.com/es/google-news.xml',
                  'https://www.eldiario.es/sitemap_google_news_25b87.xml']
    #custom_settings = {
    #    'ROBOTSTXT_OBEY': False
    #}

    namespaces = {
        'ns': 'http://www.sitemaps.org/schemas/sitemap/0.9',  # Default namespace
        'news': 'http://www.google.com/schemas/sitemap-news/0.9'
    }

    def parse(self, response):
        # Extraer datos de cada tag <url>
        for url in response.xpath('//ns:url', namespaces=self.namespaces):
            loc = url.xpath('./ns:loc/text()', namespaces=self.namespaces).get()
            title = url.xpath('./news:news/news:title/text()', namespaces=self.namespaces).get()
            publication_date = url.xpath('./news:news/news:publication_date/text()', namespaces=self.namespaces).get()
            publisher = url.xpath('./news:news/news:publication/news:name/text()', namespaces=self.namespaces).get()

            yield {
                'fuente': publisher,
                'url': loc,
                'titulo': title,
                'fecha_publicacion': publication_date,
            }
    

Overwriting sitemap_spider.py


In [10]:
copy sitemap_spider.py news_scraper\spiders

        1 archivo(s) copiado(s).


In [11]:
!scrapy crawl sitemap_spider -o news_output.csv

2025-02-15 12:42:50 [scrapy.utils.log] INFO: Scrapy 2.12.0 started (bot: news_scraper)
2025-02-15 12:42:50 [scrapy.utils.log] INFO: Versions: lxml 4.9.3.0, libxml2 2.10.3, cssselect 1.2.0, parsel 1.10.0, w3lib 2.3.1, Twisted 24.3.0, Python 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)], pyOpenSSL 25.0.0 (OpenSSL 3.3.2 3 Sep 2024), cryptography 43.0.3, Platform Windows-10-10.0.19045-SP0
2025-02-15 12:42:50 [scrapy.addons] INFO: Enabled addons:
[]
2025-02-15 12:42:50 [asyncio] DEBUG: Using selector: SelectSelector
2025-02-15 12:42:50 [scrapy.utils.log] DEBUG: Using reactor: twisted.internet.asyncioreactor.AsyncioSelectorReactor
2025-02-15 12:42:50 [scrapy.utils.log] DEBUG: Using asyncio event loop: asyncio.windows_events._WindowsSelectorEventLoop
2025-02-15 12:42:50 [scrapy.utils.log] DEBUG: Using reactor: twisted.internet.asyncioreactor.AsyncioSelectorReactor
2025-02-15 12:42:50 [scrapy.utils.log] DEBUG: Using asyncio event loop: asyncio.windows_events.

In [17]:
data_df = pd.read_csv("news_output.csv")
data_df['fecha_publicacion'] = pd.to_datetime(data_df['fecha_publicacion'], errors='coerce', utc=True)
data_df['fecha_publicacion'] = pd.to_datetime(data_df['fecha_publicacion'].dt.strftime("%Y-%m-%d"))
# fecha_inicial = pd.Timestamp("2025-2-1")
# data_df = data_df[data_df['fecha_publicacion'] >= fecha_inicial].drop_duplicates().reset_index(drop='True')
data_df

,fuente,url,titulo,fecha_publicacion
0,Radiotelevisión del Principado de Asturias,https://www.rtpa.es/noticias-sucesos/2025-02-1...,La búsqueda del desaparecido en Llanes se cent...,2025-02-15
1,Radiotelevisión del Principado de Asturias,https://www.rtpa.es/noticias-sociedad/2025-02-...,El Principado destaca el potencial del turismo...,2025-02-15
2,Radiotelevisión del Principado de Asturias,https://www.rtpa.es/noticias-asturias/2025-02-...,Asturias cierra 2024 con un aumento de los acc...,2025-02-15
3,Radiotelevisión del Principado de Asturias,https://www.rtpa.es/noticias-nacional/2025-02-...,Sanidad y las comunidades aprueban el primer p...,2025-02-14
4,Radiotelevisión del Principado de Asturias,https://www.rtpa.es/noticias-sociedad/2025-02-...,La cineasta Icíar Bollaín considera que la ult...,2025-02-14
...,...,...,...,...
6201,Europa Press,https://www.europapress.es/murcia/noticia-ayun...,El Ayuntamiento de Murcia destina más de 145.0...,NaT
6202,Europa Press,https://www.europapress.es/islas-canarias/noti...,El Gobierno de Canarias declara la prealerta p...,NaT
6203,Europa Press,https://www.europapress.es/navarra/noticia-tcc...,"TCC muestra su ""voluntad de continuar con las ...",NaT
6204,Europa Press,https://www.europapress.es/castilla-lamancha/n...,"Consejo de Gobierno aprueba el martes 3,3 mill...",NaT


In [18]:
# Articulos por fuente
data_df['fuente'].value_counts()

fuente
Europa Press                                  1000
La Razón                                       973
La Vanguardia                                  940
ABC.es                                         833
20minutos                                      606
La Nueva España                                500
ElDiario.es                                    473
<![CDATA[El Periódico]]>                       450
El Comercio: Diario de Asturias                310
Radiotelevisión del Principado de Asturias      94
Vertele                                         27
Name: count, dtype: int64

# Accedemos al contenido de las noticias

In [19]:
!pip install -q newspaper4k
!pip install -q lxml_html_clean


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: C:\Users\rebec\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: C:\Users\rebec\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [21]:
from newspaper import Article, build, Config

## Usando Newspaper

In [ ]:
config = Config()
config.browser_user_agent = 'Mozilla/5.0 (Linux; Android 10; Pixel 3 XL Build/QP1A.190711.020) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Mobile Safari/537.36'
config.request_timeout = 10
config.language= 'es'
config.request_timeout = 5
config.thread_timeout_seconds = 5
config.memoize_articles = False
config.fetch_images = False
config.follow_meta_refresh = True
config.number_threads = 16
config.headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:98.0) Gecko/20100101 Firefox/98.0",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    "Accept-Encoding": "gzip, deflate",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest": "document",
    "Sec-Fetch-Mode": "navigate",
    "Sec-Fetch-Site": "none",
    "Sec-Fetch-User": "?1",
    "Cache-Control": "max-age=0",
}

def extraer_texto_articulos(url):
    try:
        article = Article(url, config=config)
        article.download()
        article.parse()
        return article.text
    except Exception:
        print(f"Error en url: {url}")
        return None

data_df['texto'] = data_df['url'].apply(extraer_texto_articulos)

#84m 19.6s

Error en url: https://www.elcomercio.es/asturias/carnaval/charangas/carnaval-asturias-antroxu-gijon-charanga-kopavino20250215193421-nt
Error en url: https://www.larazon.es/cultura/mascarada-ancestral-zamarrones-que-puedes-perder-cantabria_2025021467af2d79500f9600010d50eb.html
Error en url: https://www.larazon.es/cultura/cine/estas-son-mejores-peliculas-romanticas-ver-san-valentin-segun_2025021467aee2f6417ec20001f95d8e.html


In [24]:
data_df

,fuente,url,titulo,fecha_publicacion,texto
0,Radiotelevisión del Principado de Asturias,https://www.rtpa.es/noticias-sucesos/2025-02-1...,La búsqueda del desaparecido en Llanes se cent...,2025-02-15,Una treintena de efectivos de la Guardia Civil...
1,Radiotelevisión del Principado de Asturias,https://www.rtpa.es/noticias-sociedad/2025-02-...,El Principado destaca el potencial del turismo...,2025-02-15,El MUJA acoge la I Jornadas de Divulgación Cie...
2,Radiotelevisión del Principado de Asturias,https://www.rtpa.es/noticias-asturias/2025-02-...,Asturias cierra 2024 con un aumento de los acc...,2025-02-15,El Principado registró el pasado año 11.765 ac...
3,Radiotelevisión del Principado de Asturias,https://www.rtpa.es/noticias-nacional/2025-02-...,Sanidad y las comunidades aprueban el primer p...,2025-02-14,Las comunidades del PP frenan el plan de salud...
4,Radiotelevisión del Principado de Asturias,https://www.rtpa.es/noticias-sociedad/2025-02-...,La cineasta Icíar Bollaín considera que la ult...,2025-02-14,La directora y guionista recibe el Galardón po...
...,...,...,...,...,...
6201,Europa Press,https://www.europapress.es/murcia/noticia-ayun...,El Ayuntamiento de Murcia destina más de 145.0...,NaT,Se trata de la rehabilitación sostenible de lo...
6202,Europa Press,https://www.europapress.es/islas-canarias/noti...,El Gobierno de Canarias declara la prealerta p...,NaT,LAS PALMAS DE GRAN CANARIA 14 Feb. (EUROPA PRE...
6203,Europa Press,https://www.europapress.es/navarra/noticia-tcc...,"TCC muestra su ""voluntad de continuar con las ...",NaT,PAMPLONA 14 Feb. (EUROPA PRESS) -\n\nMoventis ...
6204,Europa Press,https://www.europapress.es/castilla-lamancha/n...,"Consejo de Gobierno aprueba el martes 3,3 mill...",NaT,TOLEDO 14 Feb. (EUROPA PRESS) -\n\nEl Consejo ...


Extraer el contenido en paralelo

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time
data_df = data_df.iloc[:, :-1]

num_threads = 16
start_time = time.time()
with ThreadPoolExecutor(max_workers=num_threads) as executor:
    data_df['texto'] = list(executor.map(extraer_texto_articulos, data_df['url']))
print(f"--- {(time.time() - start_time):.2f}s seconds ---")
#37m 51.4s

Error en url: https://www.lne.es/internacional/2025/02/15/zelenski-pide-crear-ejercito-europeo-114332922.html
Error en url: https://www.elcomercio.es/asturias/carnaval/charangas/carnaval-asturias-antroxu-gijon-charanga-kopavino20250215193421-nt
Error en url: https://www.eldiario.es/comunitat-valenciana/comarcas/adif-adjudica-2-1-millones-obras-supresion-paso-nivel-tavernes_1_12054093.html
Error en url: https://www.larazon.es/cultura/mascarada-ancestral-zamarrones-que-puedes-perder-cantabria_2025021467af2d79500f9600010d50eb.html
Error en url: https://www.larazon.es/cultura/cine/estas-son-mejores-peliculas-romanticas-ver-san-valentin-segun_2025021467aee2f6417ec20001f95d8e.html
--- 2271.51s seconds ---


In [ ]:
from concurrent.futures import ThreadPoolExecutor
data_df = data_df.iloc[:, :-1]
num_threads = 8
with ThreadPoolExecutor(max_workers=num_threads) as executor:
    data_df['texto'] = list(executor.map(extraer_texto_articulos, data_df['url']))
#39m 2.2s

Error en url: https://www.lne.es/internacional/2025/02/15/zelenski-pide-crear-ejercito-europeo-114332922.html
Error en url: https://www.elcomercio.es/asturias/carnaval/charangas/carnaval-asturias-antroxu-gijon-charanga-kopavino20250215193421-nt
Error en url: https://www.larazon.es/cultura/mascarada-ancestral-zamarrones-que-puedes-perder-cantabria_2025021467af2d79500f9600010d50eb.html
Error en url: https://www.larazon.es/cultura/cine/estas-son-mejores-peliculas-romanticas-ver-san-valentin-segun_2025021467aee2f6417ec20001f95d8e.html


In [28]:
mascara_faltantes = data_df['texto'].isnull()
datos_faltantes = data_df[mascara_faltantes]
print(f'Articulos que no pudo obtener la librería: {len(datos_faltantes)}')

Articulos que no pudo obtener la librería: 4


Segunda pasada para obtener el contenido debido a que la primera extracción a veces falla



In [29]:
data_df.loc[mascara_faltantes, 'texto'] = data_df[mascara_faltantes]['url'].apply(extraer_texto_articulos).copy()

Error en url: https://www.lne.es/internacional/2025/02/15/zelenski-pide-crear-ejercito-europeo-114332922.html
Error en url: https://www.elcomercio.es/asturias/carnaval/charangas/carnaval-asturias-antroxu-gijon-charanga-kopavino20250215193421-nt
Error en url: https://www.larazon.es/cultura/mascarada-ancestral-zamarrones-que-puedes-perder-cantabria_2025021467af2d79500f9600010d50eb.html
Error en url: https://www.larazon.es/cultura/cine/estas-son-mejores-peliculas-romanticas-ver-san-valentin-segun_2025021467aee2f6417ec20001f95d8e.html


In [30]:
mascara_faltantes = data_df['texto'].isnull()
datos_faltantes = data_df[mascara_faltantes]
print(f'Articulos que no pudo obtener la librería: {len(datos_faltantes)}')

Articulos que no pudo obtener la librería: 4


In [31]:
data_df.head(10)

,fuente,url,titulo,fecha_publicacion,texto
0,Radiotelevisión del Principado de Asturias,https://www.rtpa.es/noticias-sucesos/2025-02-1...,La búsqueda del desaparecido en Llanes se cent...,2025-02-15,Una treintena de efectivos de la Guardia Civil...
1,Radiotelevisión del Principado de Asturias,https://www.rtpa.es/noticias-sociedad/2025-02-...,El Principado destaca el potencial del turismo...,2025-02-15,El MUJA acoge la I Jornadas de Divulgación Cie...
2,Radiotelevisión del Principado de Asturias,https://www.rtpa.es/noticias-asturias/2025-02-...,Asturias cierra 2024 con un aumento de los acc...,2025-02-15,El Principado registró el pasado año 11.765 ac...
3,Radiotelevisión del Principado de Asturias,https://www.rtpa.es/noticias-nacional/2025-02-...,Sanidad y las comunidades aprueban el primer p...,2025-02-14,Las comunidades del PP frenan el plan de salud...
4,Radiotelevisión del Principado de Asturias,https://www.rtpa.es/noticias-sociedad/2025-02-...,La cineasta Icíar Bollaín considera que la ult...,2025-02-14,La directora y guionista recibe el Galardón po...
5,Radiotelevisión del Principado de Asturias,https://www.rtpa.es/noticias-asturias/2025-02-...,Anembe apoya las medidas del Principado para c...,2025-02-14,Aconseja que se frenen los movimientos de las ...
6,Radiotelevisión del Principado de Asturias,https://www.rtpa.es/noticias-sucesos/2025-02-1...,Finaliza sin éxito y por cuarto día la búsqued...,2025-02-14,Más de 40 efectivos buscan al hombre por tierr...
7,Radiotelevisión del Principado de Asturias,https://www.rtpa.es/noticias-sociedad/2025-02-...,Piden una unidad multidisciplinar en Asturias ...,2025-02-14,Una exposición conmemora el Día de Mundial de ...
8,Radiotelevisión del Principado de Asturias,https://www.rtpa.es/noticias-economia/2025-02-...,El consorcio vasco liderado por Sidenor compra...,2025-02-14,El episodio de la venta de Talgo comenzó hace ...
9,Radiotelevisión del Principado de Asturias,https://www.rtpa.es/noticias-asturias/2025-02-...,Asturias contará con la primera ley de ciencia...,2025-02-14,Está previsto que se apruebe en el pleno del p...


Reintentos automáticamente

In [33]:
import pandas as pd
from newspaper import Article, Config
import time

# Configuración de newspaper
config = Config()
config.browser_user_agent = 'Mozilla/5.0 (Linux; Android 10; Pixel 3 XL Build/QP1A.190711.020) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Mobile Safari/537.36'
config.request_timeout = 10
config.language = 'es'
config.fetch_images = False
config.memoize_articles = False

def extraer_texto_articulo(url, max_reintentos=3, espera=2):
    """Intenta extraer el texto de un artículo con reintentos automáticos."""
    for intento in range(max_reintentos):
        try:
            article = Article(url, config=config)
            article.download()
            article.parse()
            
            # Si se obtiene texto, devolverlo
            if article.text:
                return article.text
        except Exception as e:
            print(f"Error en intento {intento + 1} para {url}: {e}")
        
        # Esperar antes de reintentar
        time.sleep(espera)

    # Si después de los reintentos sigue fallando, devolver None
    return None


# Eliminar la última columna
data_df = data_df.iloc[:, :-1]

# Número de hilos a usar
num_threads = 16

start_time = time.time()
with ThreadPoolExecutor(max_workers=num_threads) as executor:
    data_df['texto'] = list(executor.map(extraer_texto_articulo, data_df['url']))

print(f"--- Extracción completada en {(time.time() - start_time):.2f} segundos ---")

# Guardar los resultados en un nuevo CSV
data_df.to_csv("resultados.csv", index=False)
print("Archivo guardado como 'resultados.csv'.")


Error en intento 1 para https://www.lne.es/internacional/2025/02/15/zelenski-pide-crear-ejercito-europeo-114332922.html: Article `download()` failed with Status code 404 for url None on URL https://www.lne.es/internacional/2025/02/15/zelenski-pide-crear-ejercito-europeo-114332922.html
Error en intento 2 para https://www.lne.es/internacional/2025/02/15/zelenski-pide-crear-ejercito-europeo-114332922.html: Article `download()` failed with Status code 404 for url None on URL https://www.lne.es/internacional/2025/02/15/zelenski-pide-crear-ejercito-europeo-114332922.html
Error en intento 3 para https://www.lne.es/internacional/2025/02/15/zelenski-pide-crear-ejercito-europeo-114332922.html: Article `download()` failed with Status code 404 for url None on URL https://www.lne.es/internacional/2025/02/15/zelenski-pide-crear-ejercito-europeo-114332922.html
Error en intento 1 para https://www.elcomercio.es/asturias/carnaval/charangas/carnaval-asturias-antroxu-gijon-charanga-kopavino20250215193421-

## Extracción con request y palabras clave

In [ ]:
import requests
from bs4 import BeautifulSoup

headers={
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:98.0) Gecko/20100101 Firefox/98.0",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    "Accept-Encoding": "gzip, deflate",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest": "document",
    "Sec-Fetch-Mode": "navigate",
    "Sec-Fetch-Site": "none",
    "Sec-Fetch-User": "?1",
    "Cache-Control": "max-age=0",
}

def extraer_texto_articulos_request(url):
    try:
        response = requests.get(url, headers=headers )
        html_str = response.content
        soup = BeautifulSoup(html_str, 'lxml')
        # Eliminar footer
        footer = soup.find('footer')
        if footer:
            footer.decompose()

        # Encontrar todos los <a> tags
        #for a_tag in soup.find_all('a'):
        #    # Eliminar todos los parrafos dentro de <a> tag
        #    for p_tag in a_tag.find_all('p'):
        #        p_tag.decompose()
        text_content = soup.find_all('p')

        text = ''
        for i in text_content:
            text += i.text.strip() + " "
        return text
    except Exception:
        print(f"Error en url: {url}")
        return None